In [1]:
!pip install transformers accelerate torch pandas sentence-transformers pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 40.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 153.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [sentence-transformers]ence-transformers]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


----------------Imports--------------------

In [2]:
#Imports
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from pydantic import BaseModel


In [3]:
#Load Qwen 3B
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Model Loaded")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0


Model Loaded


In [30]:
import json
import re

def extract_json(text):

    match = re.search(r'\{[\s\S]*\}', text)

    if not match:
        raise ValueError("No JSON found")

    return json.loads(match.group())

In [31]:
#Pydantic models
from pydantic import BaseModel
from typing import List


class OCRResult(BaseModel):
    customer_id:int
    name:str
    dob:str
    pan:str


class IdentityResult(BaseModel):
    match_score:float
    match_status:str
    reasoning:str


class ComplianceResult(BaseModel):
    sanctions_match:bool
    pep_match:bool
    risk_level:str
    reasoning:str


class TransactionFeatures(BaseModel):
    total_credit:float
    total_debit:float
    cash_ratio:float
    avg_transaction:float
    high_value_txn_count:int


class FinancialProfile(BaseModel):
    profile:str
    risk_indicators:List[str]


class RiskResult(BaseModel):
    risk_level:str
    confidence:float
    explanation:str


class ExplainabilityResult(BaseModel):
    reasons:List[str]


class HumanReviewResult(BaseModel):
    escalate:bool
    queue:str

In [6]:
#customer synthetic data
customer = {

    "document_text": """
Name: Aarav Mehta
DOB: 12/06/1994
PAN: ABCDE1234F
""",

    "matched_name": "Aarav Mehta",

    "transactions": [

        {
            "amount":50000,
            "type":"credit",
            "mode":"salary"
        },

        {
            "amount":25000,
            "type":"debit",
            "mode":"cash"
        },

        {
            "amount":15000,
            "type":"debit",
            "mode":"cash"
        }
    ]
}

In [32]:
#using qwen
import json

def ocr_agent(document_text):

    prompt = f"""
Extract fields from document.

Document:

{document_text}

Return ONLY JSON.

{{
 "name":"",
 "dob":"",
 "pan":""
}}
"""

    response = llm(
        prompt,
        max_new_tokens=150
    )[0]["generated_text"]

    data = extract_json(response)

    return OCRResult(**data)

In [15]:
#This is where OCR and customer master meet.
from difflib import SequenceMatcher

def identity_agent(
    ocr_result,
    customer_record
):

    name_score = SequenceMatcher(
        None,
        ocr_result.name.lower(),
        customer_record["name"].lower()
    ).ratio()

    pan_match = (
        ocr_result.pan ==
        customer_record["pan"]
    )

    dob_match = (
        ocr_result.dob ==
        customer_record["dob"]
    )

    score = (
        0.5*name_score +
        0.3*int(pan_match) +
        0.2*int(dob_match)
    )

    status = (
        "MATCH"
        if score > 0.9
        else "MISMATCH"
    )

    return IdentityResult(
        match_score=score,
        match_status=status,
        reasoning=f"PAN:{pan_match}, DOB:{dob_match}"
    )

In [16]:
def compliance_agent(
    customer_name,
    pep_df,
    sanctions_df
):

    pep_match = (
        pep_df["name"]
        .str.lower()
        .eq(customer_name.lower())
        .any()
    )

    sanctions_match = (
        sanctions_df["name"]
        .str.lower()
        .eq(customer_name.lower())
        .any()
    )

    if sanctions_match:
        risk = "HIGH"

    elif pep_match:
        risk = "MEDIUM"

    else:
        risk = "LOW"

    return ComplianceResult(
        sanctions_match=sanctions_match,
        pep_match=pep_match,
        risk_level=risk,
        reasoning="Compliance screening complete"
    )

In [17]:
def transaction_feature_agent(txns):

    total_credit = sum(
        x["amount"]
        for x in txns
        if x["type"]=="credit"
    )

    total_debit = sum(
        x["amount"]
        for x in txns
        if x["type"]=="debit"
    )

    cash_count = len(
        [x for x in txns if x["mode"]=="cash"]
    )

    cash_ratio = cash_count/len(txns)

    avg_txn = (
        sum(x["amount"] for x in txns)
        / len(txns)
    )

    high_value = len(
        [
            x for x in txns
            if x["amount"] > 50000
        ]
    )

    return TransactionFeatures(
        total_credit=total_credit,
        total_debit=total_debit,
        cash_ratio=cash_ratio,
        avg_transaction=avg_txn,
        high_value_txn_count=high_value
    )

In [18]:
#Financial Profiling Agent (Qwen)
def financial_profile_agent(features):

    prompt = f"""
You are a banking analyst.

Features:

{features.model_dump_json()}

Return ONLY JSON

{{
 "profile":"",
 "risk_indicators":[]
}}
"""

    response = llm(
        prompt,
        max_new_tokens=200
    )[0]["generated_text"]

    json_part = response[
        response.find("{"):
        response.rfind("}")+1
    ]

    data = json.loads(json_part)

    return FinancialProfile(**data)

In [19]:
#risk agent(qwen)
def risk_agent(
    identity,
    compliance,
    profile,
    features
):

    prompt = f"""
Act as senior KYC officer.

Identity:
{identity.model_dump_json()}

Compliance:
{compliance.model_dump_json()}

Features:
{features.model_dump_json()}

Profile:
{profile.model_dump_json()}

Return ONLY JSON

{{
 "risk_level":"",
 "confidence":0.0,
 "explanation":""
}}
"""

    response = llm(
        prompt,
        max_new_tokens=250
    )[0]["generated_text"]

    json_part = response[
        response.find("{"):
        response.rfind("}")+1
    ]

    data = json.loads(json_part)

    return RiskResult(**data)

In [20]:
#Explainability agent
def explainability_agent(
    risk,
    compliance,
    features
):

    reasons = []

    if compliance.pep_match:
        reasons.append(
            "PEP match detected"
        )

    if compliance.sanctions_match:
        reasons.append(
            "Sanctions match detected"
        )

    if features.cash_ratio > 0.5:
        reasons.append(
            "High cash dependency"
        )

    if features.high_value_txn_count > 2:
        reasons.append(
            "Multiple high value transactions"
        )

    return ExplainabilityResult(
        reasons=reasons
    )

In [21]:
def human_review_agent(risk):

    if risk.risk_level.upper() == "HIGH":

        return HumanReviewResult(
            escalate=True,
            queue="MANUAL_REVIEW"
        )

    return HumanReviewResult(
        escalate=False,
        queue="AUTO_APPROVED"
    )

In [22]:
import pandas as pd

ocr_documents_df = pd.DataFrame([
    {
        "customer_id": 1,
        "document_text": """
Name: Aarav Mehta
DOB: 12/06/1994
PAN: ABCDE1234F
"""
    },
    {
        "customer_id": 2,
        "document_text": """
Name: Rahul Sharma
DOB: 10/02/1988
PAN: PQRST5678K
"""
    }
])

ocr_documents_df

,customer_id,document_text
0,1,\nName: Aarav Mehta\nDOB: 12/06/1994\nPAN: ABC...
1,2,\nName: Rahul Sharma\nDOB: 10/02/1988\nPAN: PQ...


In [23]:
customer_master_df = pd.DataFrame([
    {
        "customer_id": 1,
        "name": "Aarav Mehta",
        "dob": "12/06/1994",
        "pan": "ABCDE1234F",
        "occupation": "Engineer",
        "income": 1200000
    },
    {
        "customer_id": 2,
        "name": "Rahul Sharma",
        "dob": "10/02/1988",
        "pan": "PQRST5678K",
        "occupation": "Doctor",
        "income": 1800000
    }
])

customer_master_df

,customer_id,name,dob,pan,occupation,income
0,1,Aarav Mehta,12/06/1994,ABCDE1234F,Engineer,1200000
1,2,Rahul Sharma,10/02/1988,PQRST5678K,Doctor,1800000


In [24]:
transactions_df = pd.DataFrame([
    {
        "customer_id": 1,
        "amount": 50000,
        "type": "credit",
        "mode": "salary"
    },
    {
        "customer_id": 1,
        "amount": 25000,
        "type": "debit",
        "mode": "cash"
    },
    {
        "customer_id": 1,
        "amount": 15000,
        "type": "debit",
        "mode": "cash"
    },
    {
        "customer_id": 2,
        "amount": 100000,
        "type": "credit",
        "mode": "salary"
    }
])

transactions_df

,customer_id,amount,type,mode
0,1,50000,credit,salary
1,1,25000,debit,cash
2,1,15000,debit,cash
3,2,100000,credit,salary


In [25]:
pep_df = pd.DataFrame([
    {
        "name": "Minister A",
        "designation": "Cabinet Minister"
    }
])

pep_df

,name,designation
0,Minister A,Cabinet Minister


In [26]:
sanctions_df = pd.DataFrame([
    {
        "name": "Person X",
        "country": "Iran"
    }
])

sanctions_df

,name,country
0,Person X,Iran


In [27]:
# ============================
# SELECT CUSTOMER
# ============================

customer_id = 1

# ============================
# LOAD OCR DOCUMENT
# ============================

document_text = ocr_documents_df[
    ocr_documents_df["customer_id"] == customer_id
]["document_text"].iloc[0]

# ============================
# OCR AGENT
# ============================

ocr_result = ocr_agent(document_text)

print("\n========== OCR ==========")
print(ocr_result.model_dump())

# ============================
# CUSTOMER MASTER
# ============================

customer_record = customer_master_df[
    customer_master_df["customer_id"] == customer_id
].iloc[0]

# ============================
# IDENTITY AGENT
# ============================

identity_result = identity_agent(
    ocr_result,
    customer_record
)

print("\n========== IDENTITY ==========")
print(identity_result.model_dump())

# ============================
# COMPLIANCE AGENT
# ============================

compliance_result = compliance_agent(
    ocr_result.name,
    pep_df,
    sanctions_df
)

print("\n========== COMPLIANCE ==========")
print(compliance_result.model_dump())

# ============================
# TRANSACTIONS
# ============================

customer_txns = transactions_df[
    transactions_df["customer_id"] == customer_id
]

txns = customer_txns.to_dict(
    orient="records"
)

# ============================
# TRANSACTION FEATURE AGENT
# ============================

features_result = transaction_feature_agent(
    txns
)

print("\n========== FEATURES ==========")
print(features_result.model_dump())

# ============================
# FINANCIAL PROFILE AGENT
# ============================

profile_result = financial_profile_agent(
    features_result
)

print("\n========== PROFILE ==========")
print(profile_result.model_dump())

# ============================
# RISK AGENT
# ============================

risk_result = risk_agent(
    identity_result,
    compliance_result,
    profile_result,
    features_result
)

print("\n========== RISK ==========")
print(risk_result.model_dump())

# ============================
# EXPLAINABILITY AGENT
# ============================

explainability_result = explainability_agent(
    risk_result,
    compliance_result,
    features_result
)

print("\n========== EXPLAINABILITY ==========")
print(explainability_result.model_dump())

# ============================
# HUMAN REVIEW AGENT
# ============================

review_result = human_review_agent(
    risk_result
)

print("\n========== HUMAN REVIEW ==========")
print(review_result.model_dump())

# ============================
# FINAL KYC DECISION
# ============================

final_output = {

    "customer_id": customer_id,

    "ocr": ocr_result.model_dump(),

    "identity": identity_result.model_dump(),

    "compliance": compliance_result.model_dump(),

    "transaction_features":
        features_result.model_dump(),

    "financial_profile":
        profile_result.model_dump(),

    "risk":
        risk_result.model_dump(),

    "explainability":
        explainability_result.model_dump(),

    "human_review":
        review_result.model_dump()
}

print("\n========== FINAL OUTPUT ==========")

import json

print(
    json.dumps(
        final_output,
        indent=4
    )
)

JSONDecodeError: Extra data: line 6 column 1 (char 37)

In [33]:
ocr_result = ocr_agent(document_text)
print(ocr_result)

JSONDecodeError: Extra data: line 6 column 1 (char 37)